In [1]:
import torch
import pickle
from utils import (extract_full_vector, 
                   extract_subset_vector, extract_subset_vector_exp_lip, unflatten_vector, 
                   map_subset_to_full_vector, extract_subset_vector_xs_lip, extract_lip_data, 
                   reverse_extract_lip_data, extract_lip_data_seperately)
import tqdm
from collections import defaultdict
import numpy as np
import pickle
import numpy as np
from tqdm import tqdm
from collections import defaultdict
import math
import os
import yaml
from datetime import datetime

In [2]:
with open('/home/sawaiz/Documents/Lab/In_Progress/Current/Dr. Immadullah/Phase 1/code_base/pipeline/clustering_code/11_weighted_clustering_normalized_lips/pkls/full_dataset_descriptors/live_portrait_descriptor_all_with_mead.pkl', 'rb') as file:
    all_descriptors = pickle.load(file) # frame name is the key and the value is the descriptor

In [3]:
# for key in all_descriptors:
#     print(key)
#     print(all_descriptors[key])
#     break

In [7]:
# Function to extract the full 208-dimensional vector from frame data
def extract_lip_data_seperately(frame_data):
    driving_template = frame_data['driving_template_dct']
    
    motion = driving_template['motion'][0]
                     # 9 values in matrix form
    exp = motion['exp']   
    lip_vectors = []
    for lip_idx in [6, 12, 14, 17, 19, 20]:  
        lip_vectors.append(exp[:, lip_idx, :])

    lip_data = np.array(lip_vectors)

    return lip_data


In [5]:

# Dictionary to store video frames
video_dict = defaultdict(list)

# Populate the video_dict with frame arrays in order
for key, value in all_descriptors.items():
    parts = key.split('/')
    if key[0] == "M": ## mead dataset
        video_name = "/".join(parts[:-1])
        frame_number = parts[-1].split('.')[0].split("_")[-1]
    else:
        video_name = parts[1] #rawdes
        frame_number = parts[-1].split('.')[0]
    video_dict[video_name].append((frame_number, value))

# Sort video_dict by keys
video_dict = dict(sorted(video_dict.items()))

# Sort each video's frames by frame number
for video_name in video_dict:
    video_dict[video_name].sort(key=lambda x: int(x[0]))

all_vectors_full = []
for video_name, frames in tqdm(video_dict.items(), desc="Extracting vectors for clustering"):
    video_frame_vectors = []
    for frame_number, frame_data in frames:
        vector_full = extract_lip_data_seperately(frame_data)
        all_vectors_full.append(vector_full)
all_vectors_full = np.array(all_vectors_full)


Extracting vectors for clustering: 100%|██████████| 15857/15857 [05:33<00:00, 47.57it/s]  


In [6]:
all_vectors_full[0].shape

(6, 1, 3)

In [6]:
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch
# Split into train (90%) and temp (10%)
train_data, temp_data = train_test_split(all_vectors_full, test_size=0.1, random_state=42)

# Split temp into validation (5%) and test (5%)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

In [7]:
# Convert to PyTorch tensors
train_tensor = torch.tensor(train_data, dtype=torch.float32)
val_tensor = torch.tensor(val_data, dtype=torch.float32)
test_tensor = torch.tensor(test_data, dtype=torch.float32)

In [8]:
# Create TensorDatasets
train_dataset = TensorDataset(train_tensor, train_tensor)  # Input and target are the same
val_dataset = TensorDataset(val_tensor, val_tensor)
test_dataset = TensorDataset(test_tensor, test_tensor)

In [9]:
# Create DataLoaders
batch_size = 512
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [14]:
import torch
import torch.nn as nn

# Function to build a custom MLP (Multi-Layer Perceptron)
def build_mlp(layers, activation_functions):
    modules = []
    for i in range(len(layers) - 1):
        modules.append(nn.Linear(layers[i], layers[i + 1]))
        if activation_functions[i] is not None:
            modules.append(activation_functions[i]())
    return nn.Sequential(*modules)

# Encoder Module
class Encoder(nn.Module):
    def __init__(self, input_channels, coordinates_num, layer_dims, activations):
        super(Encoder, self).__init__()
        self.conv = torch.nn.Conv1d(in_channels=3,
                        out_channels=1,  # Reduce to 1 channel
                        kernel_size=1,    # Kernel size 1 to process each position independently
                        stride=1,         
                        groups=1)   
        self.encoder = build_mlp([coordinates_num] + layer_dims, activations)

    def forward(self, x):
        # x shape: (batch, coordinates, 3)
        x = x.transpose(1, 2)  # (batch, 3, coordinates)
        x = self.conv(x)  # (batch, 1, coordinates_num)
        x = x.view(x.size(0), -1)  # (batch, coordinates_num)
        encoded = self.encoder(x)
        return encoded

# Decoder Module
class Decoder(nn.Module):
    def __init__(self, latent_dim, coordinates_num, layer_dims, activations):
        super(Decoder, self).__init__()
        self.decoder = build_mlp([latent_dim] + layer_dims[:-1], activations[:-1])
        self.coordinates_num = coordinates_num
        self.output_channels = 3  # Number of channels to match original shape
        self.conv_transpose = nn.ConvTranspose1d(
            in_channels=1,
            out_channels=3,  # Output 3 channels
            kernel_size=1,   # Match encoder kernel size
            stride=1,
            groups=1
        )

    def forward(self, x):
        x = self.decoder(x)  # Through MLP layers
        x = x.view(x.size(0), 1, -1)  # Reshape for conv_transpose (batch, 1, coordinates)
        x = self.conv_transpose(x)  # (batch, 3, coordinates_num)
        x = x.transpose(1, 2)  # (batch, coordinates_num, 3)
        return x

# Autoencoder combining Encoder and Decoder
class Autoencoder(nn.Module):
    def __init__(self, encoder, decoder):
        super(Autoencoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Example configurations
input_channels = 3
coordinates_num = 6
encoded_dim = 3
encoder_layers = [512, 256, encoded_dim]  # Number of neurons in each encoder layer
encoder_activations = [nn.GELU, nn.GELU, None]  # Activation functions for encoder
decoder_layers = [256, 512, input_channels * coordinates_num]  # [256, 512, 15]
decoder_activations = [nn.GELU, nn.GELU, nn.GELU]  # Activation functions for decoder
latent_dim = encoded_dim  # Latent dimension is the output of the last encoder layer

# Initialize encoder and decoder
encoder = Encoder(input_channels=input_channels, coordinates_num=coordinates_num,
                  layer_dims=encoder_layers, activations=encoder_activations)
decoder = Decoder(latent_dim=latent_dim, coordinates_num=coordinates_num,
                  layer_dims=decoder_layers, activations=decoder_activations)

# Combine into autoencoder
autoencoder = Autoencoder(encoder=encoder, decoder=decoder)

In [15]:
# Loss and Optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.001)

In [16]:
from tqdm import tqdm

# Early stopping parameters
patience = 3  # Number of epochs to wait for improvement
threshold = 1e-6  # Minimum change in validation loss to consider as an improvement
best_val_loss = float('inf')
epochs_no_improve = 0

num_epochs = 50

for epoch in range(num_epochs):
    # Training Phase
    autoencoder.train()
    train_loss = 0
    train_steps = 0
    train_progress = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Training", leave=False)

    for inputs, _ in train_progress:
        optimizer.zero_grad()
        outputs = autoencoder(inputs)
        loss = criterion(outputs, inputs)
        loss.backward()
        optimizer.step()

        # Update cumulative average loss
        train_loss += loss.item()
        train_steps += 1
        avg_train_loss = train_loss / train_steps
        train_progress.set_postfix({"Train Loss": avg_train_loss})

    # Validation Phase
    autoencoder.eval()
    val_loss = 0
    val_steps = 0
    val_progress = tqdm(val_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Validation", leave=False)

    with torch.no_grad():
        for inputs, _ in val_progress:
            outputs = autoencoder(inputs)
            loss = criterion(outputs, inputs)

            # Update cumulative average loss
            val_loss += loss.item()
            val_steps += 1
            avg_val_loss = val_loss / val_steps
            val_progress.set_postfix({"Val Loss": avg_val_loss})

    # Print final losses for the epoch
    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {avg_train_loss:.10f}, Val Loss: {avg_val_loss:.10f}")

    # Early Stopping Check
    if avg_val_loss < best_val_loss - threshold:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"Validation loss did not improve for {epochs_no_improve} consecutive epochs.")

    if epochs_no_improve >= patience:
        print(f"Early stopping triggered at epoch {epoch+1}. Best Val Loss: {best_val_loss:.4f}")
        break


torch.Size([512, 18])


RuntimeError: Given groups=1, weight of size [1, 3, 1], expected input[1, 512, 18] to have 3 channels, but got 512 channels instead

In [94]:
autoencoder.eval()
test_loss = 0
with torch.no_grad():
    for inputs, _ in test_loader:
        outputs = autoencoder(inputs)
        loss = criterion(outputs, inputs)
        test_loss += loss.item()

print(f"Test Loss: {test_loss/len(test_loader):.10f}")
# Epoch [4/50] - Train Loss: 0.0014, Val Loss: 0.0008 ## 20 ## Test Loss: 0.0012
# Epoch [7/50] - Train Loss: 0.0010, Val Loss: 0.0008 ## 16 ## Test Loss: 0.0008
# Epoch [6/50] - Train Loss: 0.0011, Val Loss: 0.0008 ## 72 ## Test Loss: 0.0008

#latest
#Epoch [6/50] - Train Loss: 0.0003, Val Loss: 0.0002 ## Test loss: 0.0002



Test Loss: 0.0000508661


In [95]:
# Get two examples from test loader
test_iter = iter(test_loader)
single_input, _ = next(test_iter)

n_samples = 5

# Get model output and encoded embedding for these examples
with torch.no_grad():
    encoded = autoencoder.encoder(single_input)
    single_output = autoencoder(single_input)

# Print shapes
print("Input shape:", single_input.shape)
print("Encoded shape:", encoded.shape)
print("Output shape:", single_output.shape)

for sample_idx in range(n_samples):
    # Print actual vs reconstructed values for each sample side by side
    print(f"\nActual vs Reconstructed (sample {sample_idx + 1}):")
    print("Index | Original Value | Reconstructed Value | Difference")
    print("-" * 55)
    original = single_input[sample_idx].numpy().reshape(-1)
    reconstructed = single_output[sample_idx].detach().numpy().reshape(-1)
    total_diff = 0
    for i, (orig, recon) in enumerate(zip(original, reconstructed)):
        diff = abs(orig - recon)
        total_diff += diff
        print(f"{i:5d} | {orig:13.6f} | {recon:17.6f} | {diff:10.6f}")

    # Print encoded embedding values
    print(f"\nEncoded embedding for sample {sample_idx + 1}:")
    encoded_values = encoded[sample_idx].detach().numpy()
    for i, val in enumerate(encoded_values):
        print(f"Dimension {i}: {val:.6f}")

    # Calculate and print average difference
    avg_diff = total_diff / len(original)
    print(f"\nAverage absolute difference for sample {sample_idx + 1}: {avg_diff:.10f}")

    # Calculate reconstruction error for this sample
    reconstruction_error = torch.nn.functional.mse_loss(single_input[sample_idx], single_output[sample_idx])
    print(f"Reconstruction error for sample {sample_idx + 1}: {reconstruction_error:.10f}")


Input shape: torch.Size([512, 3, 6])
Encoded shape: torch.Size([512, 3])
Output shape: torch.Size([512, 3, 6])

Actual vs Reconstructed (sample 1):
Index | Original Value | Reconstructed Value | Difference
-------------------------------------------------------
    0 |     -0.000021 |         -0.001806 |   0.001785
    1 |      0.000034 |         -0.001668 |   0.001702
    2 |      0.001743 |         -0.001383 |   0.003126
    3 |      0.002424 |         -0.001672 |   0.004096
    4 |      0.003061 |          0.000422 |   0.002639
    5 |      0.001212 |         -0.001911 |   0.003124
    6 |     -0.000044 |          0.001595 |   0.001639
    7 |      0.000014 |          0.001980 |   0.001967
    8 |      0.003059 |          0.000201 |   0.002858
    9 |     -0.000808 |          0.003030 |   0.003838
   10 |      0.026215 |         -0.008312 |   0.034527
   11 |     -0.012917 |          0.004870 |   0.017787
   12 |      0.000001 |         -0.000293 |   0.000294
   13 |      0.000000 |

In [13]:
from torch.utils.data import DataLoader, TensorDataset
import torch
import numpy as np

# Convert numpy array to PyTorch tensor
input_tensor = torch.tensor(all_vectors_full, dtype=torch.float32)

# Create a DataLoader for batching
batch_size = 512
dataset = TensorDataset(input_tensor)
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

# Ensure the encoder is in evaluation mode
autoencoder.encoder.eval()

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
autoencoder.encoder.to(device)

# Container for encoded vectors
encoded_vectors_list = []

# Process the input data in batches
with torch.no_grad():
    for batch in data_loader:
        batch_inputs = batch[0].to(device)  # Get inputs and move to device
        encoded_batch = autoencoder.encoder(batch_inputs)  # Encode the batch
        encoded_vectors_list.append(encoded_batch.cpu())  # Move to CPU and store

# Concatenate all batches into a single tensor
encoded_vectors = torch.cat(encoded_vectors_list, dim=0)

# Convert back to NumPy array (if needed)
encoded_vectors_numpy = encoded_vectors.numpy()

# Print the resulting 20-dimensional vectors
print("Encoded Vectors Shape:", encoded_vectors_numpy.shape)

Encoded Vectors Shape: (1156143, 16)


In [14]:
# import pickle

# # Save the encoded vectors (NumPy array) to a pickle file
# output_file = "auto_encoder_vectors.pkl"

# with open(output_file, "wb") as f:
#     pickle.dump(encoded_vectors_numpy, f)

# print(f"Encoded vectors saved to {output_file}")

def map_subset_to_full_vector_exp_lip(full_vector, subset_output):
    c_lip = subset_output[:1]
    exp = subset_output[1:]
    full_vector[5:6] = c_lip
    full_vector[13:76] = exp
    return full_vector


In [15]:
autoencoder= autoencoder.cuda()

In [16]:
new_dict = dict()
dict_encoder_descriptors = dict()
for key, value in all_descriptors.items():
    full_vector = extract_full_vector(all_descriptors[key])
    subset_vector = extract_subset_vector(full_vector)
    torch_subset_vector = torch.tensor(subset_vector, dtype=torch.float32).cuda()
    encoder_output= autoencoder.encoder(torch_subset_vector)
    decoder_output = autoencoder.decoder(encoder_output).cpu().detach().numpy()

    encoder_output_numpy = autoencoder.encoder(torch_subset_vector).cpu().detach().numpy()
    new_dict[key] = encoder_output_numpy

    full_vector = map_subset_to_full_vector(full_vector=full_vector, subset_output=decoder_output)
    dict_encoder_descriptors[key] = unflatten_vector(full_vector)

In [17]:
import pickle
# Save the encoded vectors (NumPy array) to a pickle file
output_file = "pkls/auto_encoder_output/encoded_live_portrait_descriptor_all_with_mead.pkl"

with open(output_file, "wb") as f:
    pickle.dump(new_dict, f)

print(f"Encoded vectors saved to {output_file}")

Encoded vectors saved to pkls/auto_encoder_output/encoded_live_portrait_descriptor_all_with_mead.pkl


In [18]:
import pickle
# Save the encoded vectors (NumPy array) to a pickle file
output_file = "pkls/auto_encoder_output/autodecoded_descriptors_live_portrait_descriptor_all_with_mead.pkl"

with open(output_file, "wb") as f:
    pickle.dump(dict_encoder_descriptors, f)

print(f"Encoded vectors saved to {output_file}")

Encoded vectors saved to pkls/auto_encoder_output/autodecoded_descriptors_live_portrait_descriptor_all_with_mead.pkl


In [19]:
import torch

# File path to save the encoder
encoder_save_path = "trained_models/encoder_16_with_mead_5_148.pth"

# Save the encoder's state dictionary
torch.save(autoencoder.encoder.state_dict(), encoder_save_path)

print(f"Encoder saved successfully to {encoder_save_path}")


Encoder saved successfully to trained_models/encoder_16_with_mead_5_148.pth


In [4]:
# Create a sample numpy array of shape (6,3) with integers 1-18
import numpy as np
import torch

sample_array = np.array([
    [1, 2, 3],  # First coordinate (x,y,z)
    [4, 5, 6],  # Second coordinate
    [7, 8, 9],  # Third coordinate 
    [10, 11, 12], # Fourth coordinate
    [13, 14, 15], # Fifth coordinate
    [16, 17, 18]  # Sixth coordinate
], dtype=np.float32)
print("Original numpy array (6,3):", sample_array)

# Convert to tensor, add batch dimension and reshape to (batch=1, channels=3, length=6)
sample_tensor = torch.from_numpy(sample_array)
sample_tensor = sample_tensor.transpose(0,1).unsqueeze(0)  # Transpose and add batch dim.unsqueeze(0)  # Transpose and add batch dim
print("\nReshaped tensor (1, 3, 6):")
for i in range(3):
    print(f"Channel {i+1}:", sample_tensor[0,i,:])

# Define a 1D convolution layer that will reduce each coordinate to a single value
conv1d = torch.nn.Conv1d(in_channels=3,
                        out_channels=1,  # Reduce to 1 channel
                        kernel_size=1,    # Kernel size 1 to process each position independently
                        stride=1,         
                        groups=1)         

# Set all weights to 1/3 to average the x,y,z values and bias to 0
with torch.no_grad():
    conv1d.weight.fill_(1.0/3.0)  # Average the 3 coordinates
    conv1d.bias.fill_(0.0)

# Print the convolution weights
print("\nConvolution weights:")
print("Kernel:", conv1d.weight[0,:,:])

# Apply convolution 
output = conv1d(sample_tensor)  # Shape will be (1,1,6)

# Remove extra dimensions to get (1,6) shape
output = output.squeeze(1)  # Remove the channel dimension

print("\nFinal output (1,6):", output)

# Print shapes at each step
print("\nShape summary:")
print("Initial numpy array shape:", sample_array.shape)
print("After convolution shape:", output.shape)


Original numpy array (6,3): [[ 1.  2.  3.]
 [ 4.  5.  6.]
 [ 7.  8.  9.]
 [10. 11. 12.]
 [13. 14. 15.]
 [16. 17. 18.]]

Reshaped tensor (1, 3, 6):
Channel 1: tensor([ 1.,  4.,  7., 10., 13., 16.])
Channel 2: tensor([ 2.,  5.,  8., 11., 14., 17.])
Channel 3: tensor([ 3.,  6.,  9., 12., 15., 18.])

Convolution weights:
Kernel: tensor([[0.3333],
        [0.3333],
        [0.3333]], grad_fn=<SliceBackward0>)

Final output (1,6): tensor([[ 2.0000,  5.0000,  8.0000, 11.0000, 14.0000, 17.0000]],
       grad_fn=<SqueezeBackward1>)

Shape summary:
Initial numpy array shape: (6, 3)
After convolution shape: torch.Size([1, 6])
